In [1]:
import pandas as pd
import numpy as np
import glob

In [2]:
df = pd.read_csv('sim_ballots.csv')
df.head()

,ballot_id,1,2,3,4,5
0,0,1,4,3,2,0
1,1,1,3,4,2,0
2,2,1,3,4,2,0
3,3,1,3,4,2,0
4,4,1,3,0,4,2


In [3]:
ballots = [(list(row[1:-1]), 1) for row in df.values.tolist()]
from itertools import combinations
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        candidates.add(candidate)
candidates = list(candidates)
print(len(candidates), candidates)
def simulate_copeland_fast(ballots, candidates):
    cand_index = {c: i for i, c in enumerate(candidates)}
    n = len(candidates)
    matrix = np.zeros((n, n))
    
    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_index]
        
        # Ranked candidates vs each other: order on the ballot determines the win
        for c1, c2 in combinations(ranked, 2):
            matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Every ranked candidate beats every unranked candidate
        for c1 in ranked:
            matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Unranked vs unranked: no preference was expressed, so skip (tie)
    
    scores = {c: 0 for c in candidates}
    for c1, c2 in combinations(candidates, 2):
        i, j = cand_index[c1], cand_index[c2]
        if matrix[i][j] > matrix[j][i]:
            scores[c1] += 1
            scores[c2] -= 1
        elif matrix[j][i] > matrix[i][j]:
            scores[c2] += 1
            scores[c1] -= 1
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\nCopeland Scores:")
    for candidate, score in sorted_scores:
        print(f"  {candidate}: {score}")
    winner = sorted_scores[0][0]
    print(f"\nCopeland Winner: {winner}")
    return winner

5 [0, 1, 2, 3, 4]


In [5]:
copeland_winner = simulate_copeland_fast(ballots, candidates)


Copeland Scores:
  1: 4
  3: 2
  0: -1
  4: -2
  2: -3

Copeland Winner: 1


In [6]:
def simulate_actual_plurality_veto(ballots, candidates):
    cand_set = set(candidates)

    scores = {c: 0 for c in candidates}
    expanded_voters = []

    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        scores[ranked[0]] += weight
        for _ in range(weight):
            expanded_voters.append(ranked)

    standing = set(candidates)
    elimination_order = []

    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break

        ranked_set = set(ranked_ballot)

        
        for c in reversed(ranked_ballot):
            if c in standing:
                vetoed_this_pass = {c}
                break

        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                standing.remove(vetoed)
                elimination_order.append(vetoed)

    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    return winner, elimination_order

In [7]:
plurality_veto_winner = simulate_actual_plurality_veto(ballots, candidates)

Elimination order: [2, 4, 0, 3]
Plurality Veto Winner: 1
